# SASRec training smoke test

Run SASRec training, validation, test evaluation, and top-k inference from a notebook.
The default settings use the 500-case preprocessed sample so it can run quickly on CPU.

In [ ]:
from pathlib import Path
import sys
from types import SimpleNamespace

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

In [ ]:
import numpy as np
import torch

from src.sasrec_model import SASRec
from src.sasrec_utils import BatchSampler, evaluate, load_sasrec_dataset, recommend_topk
from src.train_sasrec import init_model, set_seed, train_one_epoch

print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())

In [ ]:
args = SimpleNamespace(
    interactions_path=str(PROJECT_ROOT / 'sample_data' / 'bpi2012_500_cases' / 'sasrec_interactions.txt'),
    output_dir=str(PROJECT_ROOT / 'outputs' / 'sasrec_notebook_500'),
    batch_size=64,
    lr=0.001,
    maxlen=50,
    hidden_units=16,
    num_blocks=1,
    num_epochs=1,
    num_heads=1,
    dropout_rate=0.2,
    l2_emb=0.0,
    device='cpu',
    norm_first=False,
    eval_every=1,
    eval_users=100,
    num_negative_samples=20,
    topk=10,
    seed=42,
)

output_dir = Path(args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
set_seed(args.seed)
dataset = load_sasrec_dataset(args.interactions_path)
user_train, user_valid, user_test, user_num, item_num = dataset
num_batch = (len(user_train) - 1) // args.batch_size + 1

print('users:', user_num)
print('items:', item_num)
print('batches:', num_batch)
print('avg_train_len:', sum(len(v) for v in user_train.values()) / len(user_train))

In [ ]:
model = init_model(user_num, item_num, args)
sampler = BatchSampler(user_train, user_num, item_num, args.batch_size, args.maxlen, args.seed)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.98))

In [ ]:
for epoch in range(1, args.num_epochs + 1):
    loss = train_one_epoch(model, sampler, optimizer, criterion, args, num_batch)
    val = evaluate(model, dataset, args, split='valid')
    test = evaluate(model, dataset, args, split='test')
    print(f'epoch={epoch}, loss={loss:.4f}')
    print(f'valid NDCG@{args.topk}: {val[0]:.4f}, HR@{args.topk}: {val[1]:.4f}')
    print(f'test  NDCG@{args.topk}: {test[0]:.4f}, HR@{args.topk}: {test[1]:.4f}')

In [ ]:
checkpoint_path = output_dir / 'sasrec_notebook_last.pth'
torch.save(model.state_dict(), checkpoint_path)
checkpoint_path

In [ ]:
recommend_user = 1
recommendations = recommend_topk(model, recommend_user, dataset, args, topk=5)
recommendations

In [ ]:
# Optional: switch to full data after the smoke test works.
# args.interactions_path = str(PROJECT_ROOT / 'data' / 'processed' / 'bpi2012_complete_only' / 'sasrec_interactions.txt')
# args.output_dir = str(PROJECT_ROOT / 'outputs' / 'sasrec_notebook_full')
# args.hidden_units = 50
# args.num_blocks = 2
# args.num_epochs = 50
# args.eval_users = 10000